# Insturctions to follow before executing this script## Step 1: Create the Isolated phase2_trellis (Run in Terminal)### Tunnel for Jupyterssh -f -N -L 5402:scn33-mn:8888 rahkus@login.npsf.cdac.in### Tunnel for Gradiossh -f -N -L 7862:scn33-mn:7860 rahkus@login.npsf.cdac.in

In [ ]:
# 1. THE FROZEN RULE: Core Config First# Keep shared-cache purging disabled unless an operator has verified no concurrent loaders.ENABLE_DISK_PURGE = Falseimport core_configprint("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")from core_config import SYSTEM_DIRS# 2. ISOLATE GPU: Lock this entire notebook to a specific GPU to prevent VRAM collisionsimport osos.environ["CUDA_VISIBLE_DEVICES"] = "0" # Change to 0, 1, 2, or 3print(f"[SYSTEM] Notebook strictly locked to GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")

In [ ]:
# Replaced the old hard-coded "kill 9 <PID>" command with a read-only process listing.# Inspect the owner/command and stop only a verified stale process manually.!ps -u "$USER" -o pid,etime,cmd | grep -E '[g]radio|[p]ipeline_trellis' || true

In [ ]:
!nvidia-smi

In [ ]:
# Verify notebook UI dependencies without upgrading packages during model setup.import importlib.metadatafor package in ("jupyter", "ipywidgets", "ipykernel"):    try:        print(f"{package}=={importlib.metadata.version(package)}")    except importlib.metadata.PackageNotFoundError:        print(f"{package}=MISSING (install only in the intended phase2_3d environment after storage preflight)")

# Step 1: TRELLIS.2 (Setup)Full C++ Compilation & DependenciesThis cell contains the complete, native compilation logic. If it detects you already built the extensions, it will safely skip to save 15 minutes. If you ever need to rebuild, change **FORCE_RECOMPILE = True**.

In [ ]:
import osimport sysimport subprocessimport shutilimport importlib.utilfrom core_config import SYSTEM_DIRSfrom storage_manager import storage_preflight_pathFORCE_RECOMPILE = False# All dependency installs/builds are performed on the compute node, but they# still consume the shared user quota. Admit the setup phase before pip/conda.setup_budget_gb = max(    1.0,    float(os.environ.get("AIEROS_TRELLIS_SETUP_BUDGET_GB", "8.0")),)ok, reason = storage_preflight_path(    SYSTEM_DIRS["cache"],    required_gb=setup_budget_gb,    require_fresh_quota=True,    minimum_free_gb=0.5,)if not ok:    raise RuntimeError(f"TRELLIS setup denied by storage policy: {reason}")def run_cmd(cmd, cwd=None):    print(f"Running: {' '.join(cmd)}")    subprocess.run(cmd, cwd=cwd, check=True)def is_installed(pkg_name):    return importlib.util.find_spec(pkg_name) is not Noneos.environ.setdefault("MAX_JOBS", "1")  # Bound native build parallelism on shared compute nodes.print("1. Verifying PyTorch 2.6.0 (CUDA 12.4)...")run_cmd([sys.executable, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0", "--index-url", "https://download.pytorch.org/whl/cu124"])print("1b. Installing CUDA-enabled xFormers matched to PyTorch 2.6.0+cu124...")run_cmd([    sys.executable, "-m", "pip", "install", "--force-reinstall",    "--no-deps", "--no-cache-dir", "xformers==0.0.29.post2",    "--index-url", "https://download.pytorch.org/whl/cu124",])print("Verifying installed xFormers CUDA kernels...")_xf = subprocess.run(    [sys.executable, "-m", "xformers.info"],    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,)print(_xf.stdout)if _xf.returncode != 0:    raise RuntimeError("xformers.info failed after the pinned wheel installation.")_xf_fields = {}for _line in _xf.stdout.splitlines():    if ":" in _line:        _key, _value = _line.split(":", 1)        _xf_fields[_key.strip()] = _value.strip()if (    _xf_fields.get("build.cuda_version") != "1204"    or _xf_fields.get("pytorch.cuda") != "available"    or _xf_fields.get("memory_efficient_attention.cutlassF-pt") != "available"):    raise RuntimeError(        "xFormers CUDA validation failed. Expected CUDA 12.4 attention kernels "        "for PyTorch 2.6.0; do not continue with TRELLIS inference."    )print("XFORMERS_CUDA_VALIDATION=PASS")print("2. Verifying ML and 3D dependencies...")run_cmd([    sys.executable, "-m", "pip", "install",     "imageio", "imageio-ffmpeg", "tqdm", "easydict", "opencv-python-headless",     "ninja", "trimesh", "transformers==4.56.1", "gradio", "tensorboard", "pandas",     "lpips", "zstandard", "kornia", "timm", "einops", "safetensors", "fire",     "accelerate", "ipywidgets"])run_cmd([sys.executable, "-m", "pip", "install", "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8"])extensions_exist = is_installed("nvdiffrast") and is_installed("nvdiffrec_render") and is_installed("cumesh") and is_installed("flex_gemm") and is_installed("o_voxel")if not FORCE_RECOMPILE and extensions_exist:    print("3. ✅ 3D C++ Extensions already detected! Skipping heavy compilation.")else:    print("3. Compiling C++ Extensions (nvdiffrast, nvdiffrec, CuMesh, FlexGEMM, o-voxel)...")    print("   (This will take 5-15 minutes. Please do not interrupt.)")    extension_budget_gb = max(1.0, float(os.environ.get("AIEROS_TRELLIS_EXTENSION_BUDGET_GB", "8.0")))    ok, reason = storage_preflight_path(        SYSTEM_DIRS["cache"],        required_gb=extension_budget_gb,        require_fresh_quota=True,        minimum_free_gb=0.5,    )    if not ok:        raise RuntimeError(f"TRELLIS extension build denied by storage policy: {reason}")    EXT_DIR = os.path.join(SYSTEM_DIRS['cache'], "trellis2_extensions")    os.makedirs(EXT_DIR, exist_ok=True)    # Nvdiffrast    nd_dir = os.path.join(EXT_DIR, "nvdiffrast")    if os.path.exists(nd_dir): shutil.rmtree(nd_dir)    run_cmd(["git", "clone", "-b", "v0.4.0", "https://github.com/NVlabs/nvdiffrast.git", nd_dir])    run_cmd([sys.executable, "-m", "pip", "install", nd_dir, "--no-build-isolation"])    # Nvdiffrec    nr_dir = os.path.join(EXT_DIR, "nvdiffrec")    if os.path.exists(nr_dir): shutil.rmtree(nr_dir)    run_cmd(["git", "clone", "-b", "renderutils", "https://github.com/JeffreyXiang/nvdiffrec.git", nr_dir])    run_cmd([sys.executable, "-m", "pip", "install", nr_dir, "--no-build-isolation"])    # CuMesh (Recursive)    cm_dir = os.path.join(EXT_DIR, "CuMesh")    if os.path.exists(cm_dir): shutil.rmtree(cm_dir)    run_cmd(["git", "clone", "https://github.com/JeffreyXiang/CuMesh.git", cm_dir, "--recursive"])    run_cmd([sys.executable, "-m", "pip", "install", cm_dir, "--no-build-isolation"])    # FlexGEMM (Recursive)    fg_dir = os.path.join(EXT_DIR, "FlexGEMM")    if os.path.exists(fg_dir): shutil.rmtree(fg_dir)    run_cmd(["git", "clone", "https://github.com/JeffreyXiang/FlexGEMM.git", fg_dir, "--recursive"])    run_cmd([sys.executable, "-m", "pip", "install", fg_dir, "--no-build-isolation"])t2_dir = os.path.join(SYSTEM_DIRS['models'], "TRELLIS.2")if FORCE_RECOMPILE or not os.path.exists(os.path.join(t2_dir, "setup.sh")):    if os.path.exists(t2_dir): shutil.rmtree(t2_dir)    print("4. Cloning Official TRELLIS.2 Repository...")    run_cmd(["git", "clone", "-b", "main", "https://github.com/microsoft/TRELLIS.2.git", t2_dir, "--recursive"])if FORCE_RECOMPILE or not is_installed("o_voxel"):    print("   Installing o-voxel...")    run_cmd([sys.executable, "-m", "pip", "install", os.path.join(t2_dir, "o-voxel"), "--no-build-isolation"])sys.path.insert(0, t2_dir)print("Environment Ready! TRELLIS.2 mapped and ready.")

# Step 3: FLUX.2 (Setup)This installs the specific diffusers patch required for FLUX.2 while protecting your PyTorch 2.6.0 environment.

In [ ]:
import sysimport subprocessprint("Installing official Diffusers 0.37.0 + compatible Hub/Transformers pins for FLUX.2 Klein...")subprocess.run([    sys.executable, "-m", "pip", "install", "-U",    "diffusers==0.37.0",    "huggingface-hub==0.36.0",    "transformers==4.56.1",], check=True)# Fail at setup time if package constraints are unresolved; don't leave this# until a user clicks Generate and receives a long lazy-import traceback.subprocess.run([    sys.executable, "-c",    (        "import importlib.metadata as m; "        "from packaging.version import Version; "        "hub=Version(m.version('huggingface-hub')); "        "assert Version('0.34.0') <= hub < Version('1.0.0'), hub; "        "from diffusers import DiffusionPipeline, Flux2KleinPipeline; "        "print('FLUX_IMPORT_CHECK=PASS', 'huggingface_hub='+str(hub), "        "'diffusers='+m.version('diffusers'))"    ),], check=True)print("FLUX.2 dependencies installed and import-checked.")from packaging.version import Version_loaded_hub = sys.modules.get("huggingface_hub")if _loaded_hub is not None:    _loaded_hub_version = Version(getattr(_loaded_hub, "__version__", "0"))    if not (Version("0.34.0") <= _loaded_hub_version < Version("1.0.0")):        raise RuntimeError(            f"FLUX dependencies were updated on disk, but this Jupyter kernel still "            f"has huggingface-hub {_loaded_hub_version} loaded in memory. "            "Restart the kernel, then execute the notebook again from the top. "            "Do not continue to TRELLIS/FLUX inference in this stale kernel."        )

# Step 2: TRELLIS.2 (Interactive Studio - Image Upload)

In [ ]:
import core_configfrom core_config import SYSTEM_DIRSimport osTRELLIS_REPO_ID = "microsoft/TRELLIS.2-4B"TRELLIS_HF_CACHE_ROOT = os.path.join(    SYSTEM_DIRS["models"], "hub", "models--microsoft--TRELLIS.2-4B")TRELLIS_SNAPSHOTS_DIR = os.path.join(TRELLIS_HF_CACHE_ROOT, "snapshots")TRELLIS_LOCAL_CACHE_AVAILABLE = (    os.path.isdir(TRELLIS_SNAPSHOTS_DIR) and bool(os.listdir(TRELLIS_SNAPSHOTS_DIR)))# Resolve the HF snapshot explicitly. Do not set global offline flags here:# the combined notebook has separate cache/admission handling for FLUX.import sysimport torchimport randomimport gcimport tracebackfrom PIL import Imageimport gradio as grfrom huggingface_hub import snapshot_downloadtry:    from huggingface_hub.errors import LocalEntryNotFoundErrorexcept ImportError:    from huggingface_hub.utils import LocalEntryNotFoundErrorfrom storage_manager import managed_storage_operationfrom storage_policy import BYTES_PER_GB, estimate_hf_repo_bytesATTN_BACKEND_REQUESTED = os.environ.get("AIEROS_TRELLIS_ATTN_BACKEND", "xformers")if ATTN_BACKEND_REQUESTED not in ("xformers", "flash_attn"):    raise ValueError(        f"Unsupported TRELLIS sparse-attention backend {ATTN_BACKEND_REQUESTED!r}; "        "use xformers or flash_attn (SDPA is not supported by TRELLIS sparse attention)."    )os.environ["ATTN_BACKEND"] = ATTN_BACKEND_REQUESTEDos.environ["SPARSE_ATTN_BACKEND"] = ATTN_BACKEND_REQUESTEDos.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'os.environ["TRELLIS_CACHE_DIR"] = os.path.join(SYSTEM_DIRS['models'], "trellis_weights")os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"# Probe the actual CUDA implementation before importing TRELLIS, so a mismatched# wheel fails here with an actionable message rather than halfway through generation.if ATTN_BACKEND_REQUESTED == "xformers":    try:        import xformers.ops as xops        probe = torch.zeros((1, 2, 1, 64), device="cuda", dtype=torch.float16)        xops.memory_efficient_attention(probe, probe, probe)        del probe        print("[ATTENTION] xFormers CUDA operator probe passed.")    except Exception as exc:        raise RuntimeError(            "The installed xFormers CUDA operator is incompatible with this PyTorch/CUDA build. "            "Run the TRELLIS setup cell in phase2_3d to install the pinned compatible wheel, "            "or set AIEROS_TRELLIS_ATTN_BACKEND=flash_attn if flash_attn is installed."        ) from excelse:    import flash_attn    print("[ATTENTION] Using flash_attn for dense and sparse attention.")from trellis2.pipelines import Trellis2ImageTo3DPipelineimport o_voxelOUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "3d_models")os.makedirs(OUTPUT_DIR, exist_ok=True)def load_trellis_pipeline():    repo_id = TRELLIS_REPO_ID    hub_cache_dir = os.path.join(SYSTEM_DIRS["models"], "hub")    try:        local_model_dir = snapshot_download(            repo_id=repo_id,            cache_dir=hub_cache_dir,            local_files_only=True,        )        print(f"TRELLIS cache hit; resolved local snapshot: {local_model_dir}")    except LocalEntryNotFoundError as cache_error:        required_gb = (            estimate_hf_repo_bytes(repo_id) / BYTES_PER_GB            + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))        )        print(            f"TRELLIS cache miss ({type(cache_error).__name__}). "            f"Storage admission budget: {required_gb:.2f} GB"        )        with managed_storage_operation(            "models",            required_gb=required_gb,            require_fresh_quota=True,            minimum_free_gb=float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")),        ):            local_model_dir = snapshot_download(                repo_id=repo_id,                cache_dir=hub_cache_dir,            )    # TRELLIS from_pretrained accepts a local path, not local_files_only.    return Trellis2ImageTo3DPipeline.from_pretrained(local_model_dir)trellis_pipe = load_trellis_pipeline()trellis_pipe.cuda()if ENABLE_DISK_PURGE and os.environ.get("AIEROS_ALLOW_SHARED_MODEL_PURGE") == "1":    print("\n[WARNING] Explicit shared-cache purge requested. This is unsafe if any other process may still be loading the same model.")    DiskPurgeManager.purge_hf_model("microsoft/TRELLIS.2-4B")else:    print("[STORAGE] Shared model/cache files retained. Set ENABLE_DISK_PURGE=True and AIEROS_ALLOW_SHARED_MODEL_PURGE=1 only after confirming no concurrent loaders.")# TRELLIS_CACHE_DIR is deliberately retained because pipeline components may use it lazily.@torch.inference_mode()def extrude_3d_asset(input_image, decimation_target: int):    if input_image is None:        raise gr.Error("Please upload an image first!")            seed = random.randint(0, 999999)    print(f"\n[EXTRUDING] Processing uploaded image | Seed: {seed}")        try:        input_image = input_image.convert("RGB")        width, height = input_image.size        max_size = 1024        if max(width, height) > max_size:            scale = max_size / float(max(width, height))            width = max(1, int(width * scale))            height = max(1, int(height * scale))            input_image = input_image.resize((width, height), Image.Resampling.LANCZOS)        # TRELLIS.2 works more predictably with a square image aligned to 64px.        square_size = max(64, ((max(width, height) + 63) // 64) * 64)        square_image = Image.new("RGB", (square_size, square_size), (255, 255, 255))        square_image.paste(input_image, ((square_size - width) // 2, (square_size - height) // 2))        input_image = square_image        torch.cuda.empty_cache()        mesh = trellis_pipe.run(input_image)[0]        mesh.simplify(16777216)        glb_path = os.path.join(OUTPUT_DIR, f"standalone_t2_{seed}.glb")        glb = o_voxel.postprocess.to_glb(            vertices            =   mesh.vertices,            faces               =   mesh.faces,            attr_volume         =   mesh.attrs,            coords              =   mesh.coords,            attr_layout         =   mesh.layout,            voxel_size          =   mesh.voxel_size,            aabb                =   [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],            decimation_target   =   int(decimation_target),            texture_size        =   4096,            remesh              =   True,            remesh_band         =   1,            remesh_project      =   0,            verbose             =   False        )        with managed_storage_operation(            "outputs",            required_gb=0.20,            require_fresh_quota=False,            minimum_free_gb=0.05,        ):            glb.export(glb_path, extension_webp=True)        del mesh, glb        gc.collect()        torch.cuda.empty_cache()        return glb_path    except Exception as exc:        error_trace = traceback.format_exc()        print(f"\\n[TRELLIS GENERATION ERROR]\\n{error_trace}")        gc.collect()        torch.cuda.empty_cache()        raise gr.Error(f"TRELLIS generation failed: {type(exc).__name__}: {exc}") from excwith gr.Blocks(title="TRELLIS.2 Standalone", theme=gr.themes.Monochrome()) as demo_trellis:    gr.Markdown("# 🕹️ TRELLIS.2-4B (Isolated Testing)")    with gr.Row():        with gr.Column(scale=1):            input_image = gr.Image(label="Upload 2D Concept", type="pil")            decimation = gr.Slider(label="Target Polygons", minimum=10000, maximum=500000, value=100000, step=10000)            generate_btn = gr.Button("Extrude to 3D", variant="primary", size="lg")        with gr.Column(scale=2):            output_3d = gr.Model3D(label="3D Interactive Viewer")    generate_btn.click(fn=extrude_3d_asset, inputs=[input_image, decimation], outputs=[output_3d])print("Starting TRELLIS Server on port 7862...")demo_trellis.launch(server_name="0.0.0.0", server_port=7862, inline=True, share=False, show_error=True)

# Step 4: FLUX.2 (Interactive Studio - Text to Image)

In [ ]:
import core_configfrom core_config import SYSTEM_DIRSfrom huggingface_hub import snapshot_downloadtry:    from huggingface_hub.errors import LocalEntryNotFoundErrorexcept ImportError:    from huggingface_hub.utils import LocalEntryNotFoundErrorimport osimport torchimport randomimport gcimport gradio as grfrom diffusers import DiffusionPipelinefrom storage_manager import managed_storage_operationfrom storage_policy import BYTES_PER_GB, estimate_hf_repo_bytesOUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "images")os.makedirs(OUTPUT_DIR, exist_ok=True)def load_flux_pipeline():    model_id = "black-forest-labs/FLUX.2-klein-4B"    hub_cache_dir = os.path.join(SYSTEM_DIRS["models"], "hub")    kwargs = dict(        torch_dtype=torch.bfloat16,        trust_remote_code=True,        text_encoder_2=None,        tokenizer_2=None,        feature_extractor=None,        image_encoder=None,    )    try:        local_model_dir = snapshot_download(            repo_id=model_id,            cache_dir=hub_cache_dir,            local_files_only=True,        )        print(f"FLUX cache hit; resolved local snapshot: {local_model_dir}")    except LocalEntryNotFoundError as cache_error:        # Only a true local cache miss enters storage admission. Don't mask import,        # compatibility, model-config, or CUDA errors as missing model files.        required_gb = (            estimate_hf_repo_bytes(model_id) / BYTES_PER_GB            + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))        )        print(            f"FLUX cache miss ({type(cache_error).__name__}). "            f"Storage admission budget: {required_gb:.2f} GB"        )        with managed_storage_operation(            "models",            required_gb=required_gb,            require_fresh_quota=True,            minimum_free_gb=float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")),        ):            local_model_dir = snapshot_download(                repo_id=model_id,                cache_dir=hub_cache_dir,            )    pipe = DiffusionPipeline.from_pretrained(local_model_dir, **kwargs)    pipe.enable_model_cpu_offload()    return pipeflux_pipe = load_flux_pipeline()@torch.inference_mode()def generate_concept(prompt: str, seed: int, randomize: bool):    if randomize: seed = random.randint(0, 999999)    print(f"\n[GENERATING] Prompt: '{prompt}' | Seed: {seed}")        generator = torch.Generator(device="cpu").manual_seed(seed)    image = flux_pipe(        prompt,        guidance_scale=3.5,         num_inference_steps=28,         max_sequence_length=512,        generator=generator    ).images[0].convert("RGB")        out_path = os.path.join(OUTPUT_DIR, f"flux2_concept_{seed}.png")    with managed_storage_operation(        "outputs",        required_gb=0.05,        require_fresh_quota=False,        minimum_free_gb=0.02,    ):        image.save(out_path)        gc.collect()    torch.cuda.empty_cache()    return image, seedwith gr.Blocks(title="FLUX.2 Standalone", theme=gr.themes.Monochrome()) as demo_flux:    gr.Markdown("# 🎨 FLUX.2-4B (Isolated Testing)")    with gr.Row():        with gr.Column(scale=1):            prompt = gr.Textbox(label="Prompt", value="A sci-fi crate", lines=4)            seed = gr.Number(label="Seed", value=42, precision=0)            randomize = gr.Checkbox(label="Randomize", value=True)            generate_btn = gr.Button("Generate", variant="primary")        with gr.Column(scale=2):            output_image = gr.Image(label="FLUX.2 Output")    generate_btn.click(fn=generate_concept, inputs=[prompt, seed, randomize], outputs=[output_image, seed])print("Starting FLUX Server on port 7863...")demo_flux.launch(server_name="0.0.0.0", server_port=7863, inline=True, share=False)

In [ ]:
# demo.close()

In [ ]:
!nvidia-smi